# Chapter 14: Joining and Concatenating

In [1]:
import polars as pl
pl.__version__  # The book is built with Polars version 1.20.0

'1.33.0'

## 14.1 Joining

`df.join(other, on, how, left/right_on=, suffix=, validate=, nulls_equal=)`<br>
Join different df by values in df with strategy

`other` the other df to be joined<br>
`on` name of join cols in both df, `left/right_on` should be `None`, `how != cross`<br>
`how` [inner(default), left, right, full, semi, anti, cross]<br>
`left/right_on` speficy name of join cols of left/right df<br>
`suffix` append to same col name<br>
`validate` check many/one to many/one relationship, default `m:m`<br>
`nulls_equal` join on null values, default never match

### 14.1.1 Join Strategies

In [2]:
df_left = pl.DataFrame({"key": ["A", "B", "C", "D"], "value": [1, 2, 3, 4]})

df_right = pl.DataFrame({"key": ["B", "C", "D", "E"], "value": [5, 6, 7, 8]})

#### 1. Inner

Only keep matched rows in both df

In [3]:
df_left.join(df_right, on="key", how="inner")

key,value,value_right
str,i64,i64
"""B""",2,5
"""C""",3,6
"""D""",4,7


#### 2. Full

Keep all rows in both df, add suffix to repeated col names

In [4]:
df_left.join(df_right, on="key", how="full", suffix="_other")

key,value,key_other,value_other
str,i64,str,i64
"""B""",2,"""B""",5
"""C""",3,"""C""",6
"""D""",4,"""D""",7
null,null,"""E""",8
"""A""",1,null,null


#### 3. Left

Only keep all rows from left df in original order. Unmatched value is null

In [5]:
df_left.join(df_right, on="key", how="left")

key,value,value_right
str,i64,i64
"""A""",1,null
"""B""",2,5
"""C""",3,6
"""D""",4,7


#### 4. Right

Only keep all rows from right df in original order, Unmatched value is null

In [6]:
df_left.join(df_right, on="key", how="right")

value,key,value_right
i64,str,i64
2,"""B""",5
3,"""C""",6
4,"""D""",7
null,"""E""",8


#### 5. Cross

Cartesian product of rows from both df, no need for `on` param

In [7]:
df_left.join(df_right, how="cross")

key,value,key_right,value_right
str,i64,str,i64
"""A""",1,"""B""",5
"""A""",1,"""C""",6
"""A""",1,"""D""",7
"""A""",1,"""E""",8
"""B""",2,"""B""",5
…,…,…,…
"""C""",3,"""E""",8
"""D""",4,"""B""",5
"""D""",4,"""C""",6


#### 6. Semi

Filter rows in left df according to successful `on` match with right df

In [8]:
df_left.join(df_right, on="key", how="semi")

key,value
str,i64
"""B""",2
"""C""",3
"""D""",4


#### 7. Anti

Contrary to Semi join, filter rows in left that DON'T match `on` right

In [9]:
df_left.join(df_right, on="key", how="anti")

key,value
str,i64
"""A""",1


### 14.1.2 Joining on Multiple Columns

`on=col_list` join on value tuple of listed cols.<br>


Join on (name, city) tuple match

In [10]:
residences_left = pl.DataFrame(
    {
        "name": ["Alice", "Bob", "Charlie", "Dave"],
        "city": ["NY", "LA", "NY", "SF"],
        "age": [25, 30, 35, 40],
    }
)

departments_right = pl.DataFrame(
    {
        "name": ["Alice", "Bob", "Charlie", "Dave"],
        "city": ["NY", "LA", "NY", "Chicago"],
        "department": ["Finance", "Marketing", "Engineering", "Operations"],
    }
)

residences_left.join(departments_right, on=["name", "city"], how="inner")

name,city,age,department
str,str,i64,str
"""Alice""","""NY""",25,"""Finance"""
"""Bob""","""LA""",30,"""Marketing"""
"""Charlie""","""NY""",35,"""Engineering"""


### 14.1.3 Validation

Validate if joining match specific type of relationship

#### 1. Many-to-many

m:m, multiple left rows MATCH multiple right rows when joining. Default

#### 2. One-to-many

1:m, one left row MATCH multiple right rows. Check if left keys are unique

#### 3. Many-to-one

m:1, multiple left rows MATCH one right row. Check if right keys are unique

#### 4. One-to-one

1:1, Check if both left and right keys are unique

Employee : department is m:1 join match, `validate='m:1'`

In [11]:
employees = pl.DataFrame(
    {
        "employee_id": [1, 2, 3, 4],
        "name": ["Alice", "Bob", "Charlie", "Dave"],
        "department_id": [10, 10, 30, 10],
    }
)

departments = pl.DataFrame(
    {
        "department_id": [10, 20, 30],
        "department_name": [
            "Information Technology",
            "Finance",
            "Human Resources",
        ],
    }
)

employees.join(departments, on="department_id", how="left", validate="m:1")

employee_id,name,department_id,department_name
i64,str,i64,str
1,"""Alice""",10,"""Information Technology"""
2,"""Bob""",10,"""Information Technology"""
3,"""Charlie""",30,"""Human Resources"""
4,"""Dave""",10,"""Information Technology"""


In [12]:
# This raises a ComputeError:
# departments = pl.DataFrame(
#     {
#         "department_id": [10, 20, 10],
#         "department_name": [
#             "Information Technology",
#             "Finance",
#             "Human Resources",
#         ],
#     }
# )

# employees.join(
#     departments, on="department_id", how="left", validate="m:1"
# )

## 14.2 Inexact Joining

`df.join_asof(other, left/right_on=, on=, by_left/right=, by=, strategy= suffix=, tolerance=, coalesce=)`<br>
Join by similar values instead of exact value match. Usually join by inexact timestamp

`other` the other df<br>
`left/right_on` left/right cols to join if different names<br>
`on` col to join if same name<br>
`by_left/right` join on these cols before asof join, to make left df more dense for asof join<br>
`by` join on these cols before as of join<br>
`strategy` [backward(default), forward, nearest]<br>
`suffix` append to cols of same name<br>
`tolerance` time or value diff tolerance<br>
`coalesce` combine join cols, default True. Auto False if join on expr

Two df join cols must be SORTED

In [13]:
df_left = pl.DataFrame({"int_id": [10, 5], "value": ["b", "a"]})

df_right = pl.DataFrame({"int_id": [4, 7, 12], "value": [1, 2, 3]})

In [14]:
# This raises an InvalidOperationError:
# df_left.join_asof(df_right, on="int_id", tolerance=3)

In [15]:
df_left = df_left.sort("int_id")
df_right = df_right

df_left.join_asof(df_right, on="int_id")

int_id,value,value_right
i64,str,i64
5,"""a""",1
10,"""b""",2


`coalesce` False shows join cols `int_id` and `int_id_right`

In [16]:
df_left.join_asof(
    df_right,
    on="int_id",
    coalesce=False,
)

int_id,value,int_id_right,value_right
i64,str,i64,i64
5,"""a""",4,1
10,"""b""",7,2


Specify `left/right_on` col names if join cols have different names

In [17]:
df_left.join_asof(
    df_right.rename({"int_id": "int_id_right"}),
    left_on="int_id",
    right_on="int_id_right",
)

int_id,value,int_id_right,value_right
i64,str,i64,i64
5,"""a""",4,1
10,"""b""",7,2


### 14.2.1 Inexact Join Strategies

`backward` default, choose right df's last record <= left record<br>
`forward` choose right df's 1st record >= left record<br>
`nearest` choose right df's nearest record to left record

In [18]:
print(df_left)
print(df_right)

shape: (2, 2)
┌────────┬───────┐
│ int_id ┆ value │
│ ---    ┆ ---   │
│ i64    ┆ str   │
╞════════╪═══════╡
│ 5      ┆ a     │
│ 10     ┆ b     │
└────────┴───────┘
shape: (3, 2)
┌────────┬───────┐
│ int_id ┆ value │
│ ---    ┆ ---   │
│ i64    ┆ i64   │
╞════════╪═══════╡
│ 4      ┆ 1     │
│ 7      ┆ 2     │
│ 12     ┆ 3     │
└────────┴───────┘


`backward` For each left `int_id`, try match last (largest) right `int_id` <= left and within `3` distance

In [19]:
df_left.join_asof(
    df_right,
    on="int_id",
    tolerance=3,
    strategy="backward",
    coalesce=False
)

int_id,value,int_id_right,value_right
i64,str,i64,i64
5,"""a""",4,1
10,"""b""",7,2


`forward` For each left `int_id`, try match 1st (smallest) right `int_id` >= left and within `3` distance

In [20]:
df_left.join_asof(
    df_right,
    on="int_id",
    tolerance=3,
    strategy="forward",
    coalesce=False
)

int_id,value,int_id_right,value_right
i64,str,i64,i64
5,"""a""",7,2
10,"""b""",12,3


`nearest` For each left `int_id`, try match nearest right `int_id` within `3` distance

In [21]:
df_left.join_asof(
    df_right,
    on="int_id",
    tolerance=3,
    strategy="nearest",
    coalesce=False
)

int_id,value,int_id_right,value_right
i64,str,i64,i64
5,"""a""",4,1
10,"""b""",12,3


### 14.2.2 Additional Fine-Tuning

Time tolerance can use `datetime.timedelta` obj or duration str like `'7d12h30m'`

If to make sure exact match, use `by` to join before asof join

### 14.2.3 Use Case: Marketing Campaign Attribution

In [22]:
campaigns = pl.scan_csv("data/campaigns.csv")
campaigns.collect()

Campaign Name,Campaign Date,Product Type
str,str,str
"""Launch""","""2023-01-01 20:00:00""","""Electronics"""
"""Discount""","""2023-04-02 00:00:00""","""Furniture"""
"""New Arrivals""","""2023-07-02 07:00:00""","""Clothing"""
"""Seasonal Sale""","""2023-10-01 04:00:00""","""Electronics"""
"""Clearance""","""2023-12-31 21:00:00""","""Books"""


In [23]:
campaigns.select(pl.col("Product Type").unique()).collect()

Product Type
str
"""Furniture"""
"""Books"""
"""Electronics"""
"""Clothing"""


In [24]:
transactions = pl.scan_csv("data/transactions.csv")
transactions.head(1).collect()

Sale Date,Product Type,Quantity
str,str,i64
"""2023-01-01 02:00:00.000000000""","""Books""",7


Transform `transactions` and `campaigns` time to the same datetime format<br>
Sort `transactions` date col, 1st join by `Product Type` then asof join `campaigns` on Date<br>
Use "backward" strategy and 60 day tolerance


In [25]:
transactions = transactions.with_columns(
    pl.col("Sale Date")
    .str.to_datetime("%Y-%m-%d %H:%M:%S%.f")
    .cast(pl.Datetime("us")),
)
campaigns = campaigns.with_columns(
    pl.col("Campaign Date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
)

sales_with_campaign_df = (
    transactions.sort("Sale Date")
    .join_asof(
        campaigns.sort("Campaign Date"),
        left_on="Sale Date",
        right_on="Campaign Date",
        by="Product Type",
        strategy="backward",
        tolerance="60d",
        check_sortedness=False,
    )
    .collect()
)
sales_with_campaign_df

Sale Date,Product Type,Quantity,Campaign Name,Campaign Date
datetime[μs],str,i64,str,datetime[μs]
2023-01-01 01:26:12.558627,"""Electronics""",2,null,null
2023-01-01 02:00:00,"""Books""",7,null,null
2023-01-01 06:14:30.703535,"""Toys""",9,null,null
2023-01-01 06:52:25.117255,"""Clothing""",9,null,null
2023-01-01 07:44:50.234511,"""Books""",7,null,null
…,…,…,…,…
2023-12-31 15:45:29.296464,"""Clothing""",10,null,null
2023-12-31 18:15:09.765488,"""Toys""",4,null,null
2023-12-31 18:33:47.441372,"""Electronics""",7,null,null


Group sales-campaign data by (product type, campaign)<br>
Find mean sales for each group, and sort by above tuple

To check if campaign can influence sales of a product type

In [26]:
(
    sales_with_campaign_df.group_by("Product Type", "Campaign Name")
    .agg(pl.col("Quantity").mean())
    .sort("Product Type", "Campaign Name")
)

Product Type,Campaign Name,Quantity
str,str,f64
"""Books""",null,5.527716
"""Clothing""",null,5.433385
"""Clothing""","""New Arrivals""",8.200581
"""Electronics""",null,5.486832
"""Electronics""","""Launch""",8.080775
"""Electronics""","""Seasonal Sale""",8.471406
"""Furniture""",null,5.430222
"""Furniture""","""Discount""",8.191888
"""Toys""",null,5.50318


Books indeed have a campaign, but no campaign sales data above?

In [27]:
campaigns.filter(pl.col("Product Type") == "Books").collect()

Campaign Name,Campaign Date,Product Type
str,datetime[μs],str
"""Clearance""",2023-12-31 21:00:00,"""Books"""


In [28]:
transactions.head(1).collect()

Sale Date,Product Type,Quantity
datetime[μs],str,i64
2023-01-01 02:00:00,"""Books""",7


In [29]:
(
    transactions.filter(
        (pl.col("Product Type") == "Books")
        & (
            pl.col("Sale Date")
            > pl.lit("2023-12-31 21:00:00").str.to_datetime()
        )
    ).collect()
)

Sale Date,Product Type,Quantity
datetime[μs],str,i64


## 14.3 Vertical and Horizontal Concatenation

3 strategies of concat:<br>
Copy data to new memory space, new df<br>
New df at current memory reference<br>
Append 2nd df to 1st

`pl.concat(items, how='vertical, rechunk=False, parallel=True)`<br>
`items` df, lf or Series to concat<br>
`how` [vertical(defailt), vertical_relaxed, horizontal, diagonal, diagonal_relaxed, align]<br>
`rechunk` result data relocate to contiguous memory. Now default False

### 14.3.1 Vertical

Vertically stack df by order<br>
`vertical_relaxed` coerce mismatched cols into their supertype

In [30]:
df1 = pl.DataFrame(
    {
        "id": [1, 2, 3],
        "value": ["a", "b", "c"],
    }
)
df2 = pl.DataFrame(
    {
        "id": [4, 5],
        "value": ["d", "e"],
    }
)
pl.concat([df1, df2], how="vertical")

id,value
i64,str
1,"""a"""
2,"""b"""
3,"""c"""
4,"""d"""
5,"""e"""


### 14.3.2 Horizontal

Horizontally concat df by order<br>
Missing rows filled by null. All col name must be unique.

In [31]:
df1 = pl.DataFrame(
    {
        "id": [1, 2, 3],
        "value": ["a", "b", "c"],
    }
)
df2 = pl.DataFrame(
    {
        "value2": ["x", "y"],
    }
)
pl.concat([df1, df2], how="horizontal")

id,value,value2
i64,str,str
1,"""a""","""x"""
2,"""b""","""y"""
3,"""c""",null


### 14.3.3 Diagonal

Concat by finding union of col names, stack cols of the same name, fill holes with null<br>
`diagonal_relaxed` coerce to supertype if type mismatch

In [32]:
df1 = pl.DataFrame(
    {
        "id": [1, 2, 3],
        "value": ["a", "b", "c"],
    }
)
df2 = pl.DataFrame(
    {
        "value": ["d", "e"],
        "value2": ["x", "y"],
    }
)
pl.concat([df1, df2], how="diagonal")

id,value,value2
i64,str,str
1,"""a""",null
2,"""b""",null
3,"""c""",null
null,"""d""","""x"""
null,"""e""","""y"""


### 14.3.4 Align

Align common col's common values, fill holes with null

In [33]:
df1 = pl.DataFrame(
    {
        "id": [1, 2, 3],
        "value": ["a", "b", "c"],
    }
)
df2 = pl.DataFrame(
    {
        "value": ["a", "c", "d"],
        "value2": ["x", "y", "z"],
    }
)
pl.concat([df1, df2], how="align")

id,value,value2
i64,str,str
1,"""a""","""x"""
2,"""b""",null
3,"""c""","""y"""
null,"""d""","""z"""


`pl.align_frames()` provides richer customization of df alignment concat<br>
df misissing a value -> filled with null<br>
df repeated an alignment value -> Cartesian product

In [43]:
df1 = pl.DataFrame(
    {
        "id": [1, 2, 2],
        "value": ["a", "c", "b"],
    }
)
df2 = pl.DataFrame(
    {
        "id": [2, 2],
        "value": ["x", "y"],
    }
)
pl.align_frames(df1, df2, on="id")

[shape: (5, 2)
 ┌─────┬───────┐
 │ id  ┆ value │
 │ --- ┆ ---   │
 │ i64 ┆ str   │
 ╞═════╪═══════╡
 │ 1   ┆ a     │
 │ 2   ┆ c     │
 │ 2   ┆ b     │
 │ 2   ┆ c     │
 │ 2   ┆ b     │
 └─────┴───────┘,
 shape: (5, 2)
 ┌─────┬───────┐
 │ id  ┆ value │
 │ --- ┆ ---   │
 │ i64 ┆ str   │
 ╞═════╪═══════╡
 │ 1   ┆ null  │
 │ 2   ┆ x     │
 │ 2   ┆ x     │
 │ 2   ┆ y     │
 │ 2   ┆ y     │
 └─────┴───────┘]

### 14.3.5 Relaxed

Available for "vertical", "horizontal", "diagonal"<br>
Cols of the same name with different datatype, will transform into supertype:<br>
int32 match int64 -> int64<br>
int match float -> float<br>
int match str -> str<br>

In [36]:
df1 = pl.DataFrame(
    {
        "id": [1, 2, 3],
        "value": ["a", "b", "c"],
    }
)
df2 = pl.DataFrame(
    {
        "id": [4.0, 5.0],
        "value": [1, 2],
    }
)
# This raises a SchemaError:
# pl.concat([df1, df2], how="vertical")

`id` col coerced into float64, `value` col coerced into str

In [37]:
pl.concat([df1, df2], how="vertical_relaxed")

id,value
f64,str
1.0,"""a"""
2.0,"""b"""
3.0,"""c"""
4.0,"""1"""
5.0,"""2"""


### 14.3.6 Stacking

`df.vstack()`, `df.hstack()` vertical/horizontal stack df without rechunk<br>
`pl.concat()` vertically uses `df.vstack()` if choose no rechunk<br>
Support DF but not LF

 `df.vstack()` requires same col count, name and dtype

In [38]:
df1 = pl.DataFrame(
    {
        "id": [1, 2],
        "value": ["a", "b"],
    }
)
df2 = pl.DataFrame(
    {
        "id": [3, 4],
        "value": ["c", "d"],
    }
)
df1.vstack(df2)

id,value
i64,str
1,"""a"""
2,"""b"""
3,"""c"""
4,"""d"""


`df.hstack()` requires same row count

In [39]:
df1 = pl.DataFrame(
    {
        "id": [1, 2],
        "value": ["a", "b"],
    }
)
df2 = pl.DataFrame(
    {
        "value2": ["x", "y"],
    }
)
df1.hstack(df2)

id,value,value2
i64,str,str
1,"""a""","""x"""
2,"""b""","""y"""


### 14.3.7 Appending

`series.append()` append one series after another, and keep left series col names

In [40]:
series_a = pl.Series("a", [1, 2])
series_b = pl.Series("b", [3, 4])
series_a.append(series_b)

a
i64
1
2
3
4


### 14.3.8 Extending

`df.extend()` good for copying small df after a big df, memory contiguous

In [41]:
df1 = pl.DataFrame(
    {
        "id": [1, 2],
        "value": ["a", "b"],
    }
)
df2 = pl.DataFrame(
    {
        "id": [3, 4],
        "value": ["c", "d"],
    }
)
df1.extend(df2)

id,value
i64,str
1,"""a"""
2,"""b"""
3,"""c"""
4,"""d"""


## Takeaways